# Satellite Image Classification - Pro Max Edition
**Objective:** 90% F1-Score in < 1 Hour

This notebook is optimized for Kaggle GPU environments using:
- **EfficientNet-B0** (Pre-trained)
- **Automatic Mixed Precision (AMP)** for 2x speed
- **OneCycleLR** Scheduler for fast convergence
- **Label Smoothing** for better generalization

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
import glob

# Detect Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Training on: {DEVICE}")

In [ ]:
# --- AUTOMATIC PATH DETECTION ---
def find_base_path():
    # Try Kaggle Input first
    kaggle_paths = glob.glob('/kaggle/input/**/Train.csv', recursive=True)
    if kaggle_paths:
        return os.path.dirname(kaggle_paths[0])
    
    # Try Local workspace second
    local_paths = glob.glob('./**/Train.csv', recursive=True)
    if local_paths:
        return os.path.dirname(local_paths[0])
        
    return "./main_dataset/DATASET" # Final Fallback

BASE_PATH = find_base_path()
print(f"📂 Detected Base Path: {BASE_PATH}")
if not os.path.exists(os.path.join(BASE_PATH, 'Train.csv')):
    print("❌ ERROR: Train.csv not found! Check if the dataset is correctly attached.")
    print("Available in /kaggle/input:", os.listdir('/kaggle/input') if os.path.exists('/kaggle/input') else 'None')

# --- HYPERPARAMETERS ---
IMG_SIZE = 224             
BATCH_SIZE = 48            
EPOCHS = 20                
LEARNING_RATE = 1e-3       
NUM_CLASSES = 10           

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, df, root_dir, transform=None, is_test=False, label_map=None):
        self.df = df
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test
        self.label_map = label_map

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx, 0] # IMAGE column
        img_path = os.path.join(self.root_dir, img_name)
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (IMG_SIZE, IMG_SIZE))
            
        if self.transform: image = self.transform(image)
        if self.is_test: return image
        
        label_name = self.df.iloc[idx, 1] # LABEL column
        label = self.label_map[label_name] if self.label_map else int(label_name)
        return image, label

In [ ]:
# --- EDA: Exploratory Data Analysis ---
train_csv_path = os.path.join(BASE_PATH, 'Train.csv')
if os.path.exists(train_csv_path):
    eda_df = pd.read_csv(train_csv_path)
    
    # 1. Distribution Plot
    plt.figure(figsize=(15, 5))
    sns.countplot(data=eda_df, x='LABEL', order=sorted(eda_df['LABEL'].unique()), palette='viridis')
    plt.title('Training Class Distribution')
    plt.xticks(rotation=45)
    plt.show()
    
    # 2. Random Grid Plot
    # Logic to handle both train_images and Train_Images case sensitivity
    possible_dirs = ['train_images', 'Train_Images', 'Train_images']
    actual_dir = 'train_images'
    for d in possible_dirs:
        if os.path.exists(os.path.join(BASE_PATH, d)):
            actual_dir = d
            break
            
    temp_ds = ImageDataset(eda_df.sample(16), os.path.join(BASE_PATH, actual_dir))
    plt.figure(figsize=(12, 12))
    for i in range(16):
        img, lbl = temp_ds[i]
        plt.subplot(4, 4, i+1)
        plt.imshow(img)
        plt.title(lbl)
        plt.axis('off')
    plt.tight_layout()
    plt.suptitle('Random Training Samples', y=1.02, fontsize=16)
    plt.show()
else:
    print("EDA Skip: Train.csv not found at", train_csv_path)

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(180),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
# --- Training Logic ---
def run_training():
    # 1. Prepare Data
    train_df_full = pd.read_csv(os.path.join(BASE_PATH, 'Train.csv'))
    unique_labels = sorted(train_df_full['LABEL'].unique())
    label_map = {name: i for i, name in enumerate(unique_labels)}
    inv_label_map = {i: name for name, i in label_map.items()}
    
    train_data, val_data = train_test_split(train_df_full, test_size=0.15, stratify=train_df_full['LABEL'], random_state=42)

    # Determine correct image directory names
    train_img_dir = 'train_images' if os.path.exists(os.path.join(BASE_PATH, 'train_images')) else 'Train_Images'
    
    train_ds = ImageDataset(train_data, os.path.join(BASE_PATH, train_img_dir), transform=train_transform, label_map=label_map)
    val_ds = ImageDataset(val_data, os.path.join(BASE_PATH, train_img_dir), transform=val_transform, label_map=label_map)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

    # 2. Build Model
    model = models.efficientnet_b0(weights='DEFAULT')
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
    model = model.to(DEVICE)

    # 3. Optimization
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LEARNING_RATE, steps_per_epoch=len(train_loader), epochs=EPOCHS)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = torch.cuda.amp.GradScaler()

    history = {'train_acc': [], 'val_acc': [], 'val_f1': []}
    best_f1 = 0.0

    for epoch in range(EPOCHS):
        model.train()
        train_correct, train_total = 0, 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
        for images, labels in pbar:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
            pbar.set_postfix({'acc': f"{100*train_correct/train_total:.2f}%"})

        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                outputs = model(images.to(DEVICE))
                _, predicted = torch.max(outputs.data, 1)
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.numpy())
        
        acc = accuracy_score(all_labels, all_preds) * 100
        f1 = f1_score(all_labels, all_preds, average='macro') * 100
        print(f"⭐ Validation Acc: {acc:.2f}% | F1: {f1:.2f}%")
        history['val_acc'].append(acc)
        history['val_f1'].append(f1)
        if f1 > best_f1:
            best_f1 = f1
            torch.save({'model': model.state_dict(), 'inv_map': inv_label_map}, 'best_model.pth')
            print("🔥 New Best Model Saved!")
    return model, val_loader, inv_label_map, history

model, val_loader, inv_label_map, history = run_training()

In [ ]:
# --- Evaluation Visualization ---
plt.figure(figsize=(10, 4))
plt.plot(history['val_acc'], label='Accuracy', marker='o')
plt.plot(history['val_f1'], label='F1-Score', marker='s')
plt.axhline(90, color='red', linestyle='--', label='Target 90%')
plt.title('Validation Performance Over Time')
plt.xlabel('Epoch')
plt.ylabel('%')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# --- Confusion Matrix ---
class_labels = sorted(inv_label_map.values())
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Final Matrix"):
        outputs = model(images.to(DEVICE))
        _, predicted = torch.max(outputs.data, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)
disp.plot(cmap='Blues', ax=plt.gca(), xticks_rotation='vertical')
plt.show()

In [ ]:
print("🏁 Generating FINAL.csv for Web Platform...")
checkpoint = torch.load('best_model.pth')
model.load_state_dict(checkpoint['model'])
model.eval()

test_df = pd.read_csv(os.path.join(BASE_PATH, 'Test.csv'))
test_img_dir = 'test_images' if os.path.exists(os.path.join(BASE_PATH, 'test_images')) else 'Test_Images'

test_ds = ImageDataset(test_df, os.path.join(BASE_PATH, test_img_dir), transform=val_transform, is_test=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

preds = []
with torch.no_grad():
    for images in tqdm(test_loader):
        outputs = model(images.to(DEVICE))
        _, p = torch.max(outputs, 1)
        preds.extend(p.cpu().numpy())

pd.DataFrame({'IMAGE': test_df['IMAGE'], 'LABEL': preds}).to_csv('FINAL.csv', index=False)
print("✅ FINAL.csv created successfully with Integer Labels. Ready for Submit!")